
# Inhibitory modulation of dynamics along a connectomic backbone

This notebook analyzes how negative/inhibitory connectivity modulates dynamics around a fixed directed backbone.

The dynamical model is

\[
\dot{\mathbf x}=A(g)\mathbf x+\mathbf u(t),
\qquad
A(g)=-\alpha I+\beta\left(W_E-gW_I\right),
\]

where \(W_E\ge 0\) contains excitatory weights, \(W_I\ge 0\) contains inhibitory magnitudes, \(g\) scales inhibition, \(\alpha\) is intrinsic decay, and \(\beta\) sets global coupling.

The notebook follows this workflow:

1. Load and validate the signed directed matrix.
2. Convert it to the state-space orientation \(W_{\text{target},\text{source}}\).
3. Define a fixed backbone ordering.
4. Split excitation and inhibition.
5. Inject an impulse at the start of the backbone.
6. Measure propagation amplitude, integrated activity, and arrival time.
7. Sweep inhibitory strength \(g\).
8. Examine eigenvalue stability and transient amplification.
9. Ablate inhibitory neurons and inhibitory edges.
10. Classify inhibitory edges as feedforward/feedback relative to backbone position.
11. Compare the observed inhibitory organization with randomized nulls.

> **Backbone ordering:** this notebook now loads the feedback-arc-minimizing ordering written by `ee_backbone_method_comparison.ipynb` to `ee_backbone_comparison_outputs/backbone_ordering.csv` (Vahidi 2025). That ordering covers the 32 excitatory types; the remaining inhibitory types are appended after it, so motif displacement is measured against excitatory backbone position. If that file is absent the notebook falls back to CSV label order and prints a warning, in which case results are not biologically interpretable.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from ei_backbone_analysis_helpers import (
    backbone_metrics,
    dale_sender_summary,
    inhibition_strength_sweep,
    inhibitory_backbone_motifs,
    inhibitory_edge_ablation,
    inhibitory_node_ablation,
    inhibitory_target_null_model,
    impulse_response,
    load_signed_matrix,
    normalize_and_split,
    resolve_backbone,
    system_matrix,
    transient_amplification,
    weight_scale_report,
)

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.max_rows", 100)


## 1. Load and validate the connectomic matrix

In [ ]:
MATRIX_PATH = Path("matrices/mij_matrix.csv")

# Orientation of the CSV:
# "source_rows"  -> CSV entry [i,j] means source i -> target j
# "target_rows"  -> CSV entry [i,j] means source j -> target i
CSV_ORIENTATION = "source_rows"

labels, W_csv, W_state_raw, summary = load_signed_matrix(
    MATRIX_PATH,
    csv_orientation=CSV_ORIENTATION,
    zero_diagonal=True,
)
n = len(labels)

display(summary)



### State-space orientation

Throughout the dynamics below, the convention is

\[
\dot x_i=\sum_j A_{ij}x_j,
\]

so **column \(j\)** is the source and **row \(i\)** is the target. Thus \(W_{ij}\) means \(j\to i\).

If your CSV instead stores source neurons in rows, the matrix must be transposed once.


In [ ]:
display(weight_scale_report(W_state_raw))


## 2. Define the backbone

In [ ]:
# The backbone ordering produced by `ee_backbone_method_comparison.ipynb`
# (feedback-arc minimization, Vahidi 2025). Falls back to the CSV label-order
# placeholder only if that notebook has not been run yet.
ORDERING_CSV = Path("ee_backbone_comparison_outputs/backbone_ordering.csv")

if ORDERING_CSV.exists():
    BACKBONE_LABELS = pd.read_csv(ORDERING_CSV)["node"].tolist()
    print(f"Loaded backbone ordering from {ORDERING_CSV} "
          f"({len(BACKBONE_LABELS)} excitatory types).")
else:
    BACKBONE_LABELS = None
    print(f"{ORDERING_CSV} not found; falling back to CSV label order.")

# The E-to-E ordering covers only the excitatory types. The full matrix has 85
# nodes, so inhibitory types are appended after the excitatory backbone; motif
# displacement is measured against excitatory backbone position, which is what
# the feedforward/feedback classification is about.
if BACKBONE_LABELS is not None:
    missing = [x for x in BACKBONE_LABELS if x not in labels]
    if missing:
        raise ValueError(f"Ordering labels absent from the full matrix: {missing[:5]}")
    BACKBONE_LABELS = BACKBONE_LABELS + [x for x in labels if x not in set(BACKBONE_LABELS)]

backbone, idx, backbone_idx, position, used_placeholder_backbone = resolve_backbone(
    labels,
    BACKBONE_LABELS,
)

if used_placeholder_backbone:
    print("WARNING: using CSV label order as a placeholder backbone.")

print(f"Backbone length: {len(backbone)}")
print("First 10 nodes:", backbone[:10])



## 3. Normalize scale and split excitation/inhibition

Because the supplied matrix spans a large numerical range, raw weights can make the linear system numerically explosive. We therefore separate **topology/sign** from an explicit global coupling parameter.

The normalized connectivity is

\[
\widehat W=\frac{W}{s},
\]

where \(s\) is the 95th percentile of nonzero \(|W_{ij}|\). This is robust to very large outliers. You can change the scaling rule if the absolute units of your weights are already dynamically meaningful.


In [ ]:
SCALE, W, W_E, W_I = normalize_and_split(W_state_raw, quantile=0.95)

ALPHA = 1.0   # intrinsic decay/leak
BETA = 0.8    # global network coupling

print(f"Normalization scale = {SCALE:.4g}")
print(f"Excitatory edges = {(W_E > 0).sum()}")
print(f"Inhibitory edges = {(W_I > 0).sum()}")


### Optional Dale-law diagnostic

In [ ]:
dale = dale_sender_summary(labels, W)
display(dale["classification"].value_counts())
display(dale.sort_values("fraction_negative", ascending=False).head(15))



## 4. Impulse propagation along the backbone

We stimulate the first backbone node with a unit impulse. For an impulse at \(t=0\),

\[
x(t)=e^{A(g)t}e_{v_1}.
\]

For each backbone node we compute:

- **peak amplitude**
- **integrated absolute activity**
- **time of peak**
- **inhibition index** relative to \(g=0\)

\[
I_k(g)=1-\frac{\max_t |x_k^{(g)}(t)|}
{\max_t |x_k^{(0)}(t)|+\varepsilon}.
\]


In [ ]:
T_MAX = 12.0
N_TIME = 241
times = np.linspace(0.0, T_MAX, N_TIME)

start_node = backbone[0]
start_i = idx[start_node]
x0 = np.zeros(n)
x0[start_i] = 1.0

X0 = impulse_response(W_E, W_I, x0, times, g=0.0, alpha=ALPHA, beta=BETA)
X1 = impulse_response(W_E, W_I, x0, times, g=1.0, alpha=ALPHA, beta=BETA)

m0 = backbone_metrics(X0, times, backbone, backbone_idx)
m1 = backbone_metrics(X1, times, backbone, backbone_idx)
m1["inhibition_index_vs_g0"] = 1 - m1["peak_abs"].to_numpy() / (m0["peak_abs"].to_numpy() + 1e-12)

display(m1.head(15))


In [ ]:

plt.figure(figsize=(10, 4))
plt.plot(m0["backbone_position"], m0["peak_abs"], label="g=0")
plt.plot(m1["backbone_position"], m1["peak_abs"], label="g=1")
plt.xlabel("Backbone position")
plt.ylabel("Peak |activity|")
plt.title("Peak impulse propagation along backbone")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(m1["backbone_position"], m1["inhibition_index_vs_g0"])
plt.axhline(0, linewidth=1)
plt.xlabel("Backbone position")
plt.ylabel("Inhibition index")
plt.title("Suppression/amplification relative to no inhibition")
plt.tight_layout()
plt.show()


## 5. Sweep inhibitory strength

In [ ]:
G_VALUES = np.linspace(0.0, 2.0, 21)
terminal_i = backbone_idx[-1]

sweep = inhibition_strength_sweep(
    G_VALUES,
    W_E,
    W_I,
    x0,
    times,
    terminal_i,
    backbone_idx,
    alpha=ALPHA,
    beta=BETA,
)
display(sweep)


In [ ]:

plt.figure(figsize=(7, 4))
plt.plot(sweep["g"], sweep["terminal_peak"], marker="o")
plt.xlabel("Inhibition scale g")
plt.ylabel("Terminal peak |activity|")
plt.title("Backbone transmission vs inhibition")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(sweep["g"], sweep["spectral_abscissa"], marker="o")
plt.axhline(0, linewidth=1)
plt.xlabel("Inhibition scale g")
plt.ylabel(r"$\max \Re(\lambda(A))$")
plt.title("Linear stability vs inhibition")
plt.tight_layout()
plt.show()



## 6. Global transient amplification

Eigenvalues describe asymptotic stability, but a directed non-normal network can transiently amplify perturbations even when all eigenvalues have negative real part.

We compute

\[
G(t)=\|e^{A t}\|_2,
\qquad
G_{\max}=\max_t G(t).
\]

This exact calculation is more expensive, so it is evaluated only for selected \(g\) values on a coarse time grid.

The single sharpest diagnostic here is the pair (spectral abscissa, numerical abscissa):

$$\alpha(A)=\max\Re\lambda(A),\qquad \omega(A)=\lambda_{\max}\!\left(\tfrac{A+A^{\mathsf T}}{2}\right).$$

\(\alpha(A)<0\) means every eigendirection decays, so the system is asymptotically stable and any amplification seen is genuinely transient rather than truncated exponential growth. \(\omega(A)>0\) is the initial growth rate of \(\|x\|\) and is positive exactly when the system is *reactive*. A large gap between the two is the standard quantitative signature of non-normality, and it is a stronger statement than \(G_{\max}\) alone. Both are computed below and both should be reported.


In [ ]:
TRANSIENT_G = [0.0, 0.5, 1.0, 1.5, 2.0]
TRANSIENT_TIMES = np.linspace(0.0, 8.0, 41)

transient = transient_amplification(
    TRANSIENT_G,
    TRANSIENT_TIMES,
    W_E,
    W_I,
    alpha=ALPHA,
    beta=BETA,
)
display(transient)

print(
    f"Spectral abscissa ranges [{transient['spectral_abscissa'].min():.4f}, "
    f"{transient['spectral_abscissa'].max():.4f}] -> "
    + ("stable at every g tested; G_max is genuine transient amplification."
       if transient["spectral_abscissa"].max() < 0 else
       "UNSTABLE for some g; G_max over a finite window is NOT transient amplification there.")
)
print(
    f"Numerical abscissa ranges [{transient['numerical_abscissa'].min():.4f}, "
    f"{transient['numerical_abscissa'].max():.4f}] "
    f"-> spectral/numerical gap ~ {transient['numerical_abscissa'].max() - transient['spectral_abscissa'].max():.2f}, "
    "the non-normality signature."
)

plt.figure(figsize=(7, 4))
plt.plot(transient["g"], transient["G_max"], marker="o")
plt.xlabel("Inhibition scale g")
plt.ylabel(r"$G_{\max}$")
plt.title("Maximum global transient amplification")
plt.tight_layout()
plt.show()



## 7. Inhibitory-neuron ablation

For each source neuron \(j\), remove its outgoing inhibitory effects by zeroing column \(j\) of \(W_I\), then measure the change in terminal peak transmission:

\[
\Delta_j =
F(W_I^{(-j)})-F(W_I).
\]

Positive \(\Delta_j\) means removing that neuron's inhibition increases terminal propagation, so the intact neuron suppresses the backbone endpoint.


In [ ]:
ABLATION_G = 1.0
baseline_X = impulse_response(W_E, W_I, x0, times, g=ABLATION_G, alpha=ALPHA, beta=BETA)
baseline_terminal_peak = np.abs(baseline_X[:, terminal_i]).max()

node_ablation = inhibitory_node_ablation(
    labels,
    W_E,
    W_I,
    x0,
    times,
    terminal_i,
    position,
    baseline_terminal_peak,
    g=ABLATION_G,
    alpha=ALPHA,
    beta=BETA,
)

display(node_ablation.head(20))


In [ ]:

top = node_ablation.head(20).iloc[::-1]
plt.figure(figsize=(8, 7))
plt.barh(top["node"], top["delta_terminal_peak"])
plt.xlabel("Change in terminal peak after inhibitory ablation")
plt.title("Most influential inhibitory source neurons")
plt.tight_layout()
plt.show()


## 8. Inhibitory-edge ablation

In [ ]:
# Finite-difference edge ablation is expensive. Analyze the strongest inhibitory edges first.
TOP_K_EDGE_ABLATIONS = 50

edge_ablation = inhibitory_edge_ablation(
    labels,
    W_E,
    W_I,
    x0,
    times,
    terminal_i,
    baseline_terminal_peak,
    top_k=TOP_K_EDGE_ABLATIONS,
    g=ABLATION_G,
    alpha=ALPHA,
    beta=BETA,
)
display(edge_ablation.head(20))



## 9. Inhibitory motifs relative to the backbone

Define the backbone coordinate \(p(v_k)=k\). For an inhibitory edge \(j\to i\),

\[
d_{ji}=p(i)-p(j).
\]

- \(d>0\): forward/feedforward relative to the backbone
- \(d<0\): backward/feedback relative to the backbone
- large \(|d|\): long-range relative to the backbone


In [ ]:
motifs = inhibitory_backbone_motifs(labels, W_I, position)
display(motifs.groupby("motif").agg(
    n_edges=("motif", "size"),
    mean_magnitude=("inhibitory_magnitude", "mean"),
    median_distance=("distance", "median"),
    mean_abs_distance=("abs_distance", "mean"),
))


In [ ]:

plt.figure(figsize=(8, 4))
for name, grp in motifs.groupby("motif"):
    plt.hist(grp["distance"], bins=30, alpha=0.5, label=name)
plt.axvline(0, linewidth=1)
plt.xlabel("Backbone displacement: target position - source position")
plt.ylabel("Count")
plt.title("Spatial organization of inhibitory edges")
plt.legend()
plt.tight_layout()
plt.show()



## 10. Null model for inhibitory placement

The following null preserves, for each source neuron:

- its number of inhibitory outgoing edges,
- its multiset of inhibitory weights,
- its identity as an inhibitory source,

while randomizing inhibitory **targets**.

It does **not** preserve inhibitory target in-degree, so interpret this as a first-order spatial-placement null rather than a full directed degree-preserving rewiring. A strength-preserving randomization (Milisav et al., 2024, *Nature Computational Science*, which handles signed and directed networks) would be the stronger comparison.

**Inference is by empirical p-value, not by z-score.** This null is strongly right-skewed: a minority of randomizations produce very large terminal peaks, so the null mean sits far above the null median and the standard deviation is several times the mean. A z-score against a distribution like that is not interpretable. At the previous default of `N_NULL = 25` it was not even reproducible — resampling 25 draws repeatedly from a larger null gives z values ranging from roughly -0.2 to +34. The estimator used now is

$$p=\frac{1+\#\{\text{null}\ge\text{observed}\}}{1+n_{\text{null}}},$$

matching the convention prescribed in the null-model section of `ee_backbone_deterministic_stochastic_comparison.ipynb`, and `N_NULL` is raised to 500. The spectral abscissa of every null draw is also recorded, because a randomization that destabilized the system would inflate the terminal peak for reasons unrelated to inhibitory placement.


In [ ]:
N_NULL = 500
NULL_G = 1.0
RANDOM_SEED = 2026

observed_metric = baseline_terminal_peak
null_metrics, null_summary = inhibitory_target_null_model(
    W_E,
    W_I,
    x0,
    times,
    terminal_i,
    observed_metric,
    n_null=N_NULL,
    g=NULL_G,
    random_seed=RANDOM_SEED,
    alpha=ALPHA,
    beta=BETA,
)

display(null_summary)
print(
    f"Observed sits at the {null_summary['observed_percentile_in_null']:.1f}th "
    f"percentile of the null; empirical p (observed >= null) = "
    f"{null_summary['empirical_p_observed_ge_null']:.4f}"
)
if null_summary["n_null_unstable"] > 0:
    print(f"WARNING: {int(null_summary['n_null_unstable'])} null draw(s) were unstable.")

plt.figure(figsize=(7, 4))
plt.hist(null_metrics, bins=12)
plt.axvline(observed_metric, linewidth=2, label="observed")
plt.xlabel("Terminal peak |activity|")
plt.ylabel("Null count")
plt.axvline(null_summary["null_median"], linewidth=1, linestyle="--", label="null median")
plt.title("Observed inhibitory organization vs randomized targets")
plt.legend()
plt.tight_layout()
plt.show()



## 11. Suggested interpretation checklist

Use the analyses jointly rather than interpreting a single statistic.

- **Propagation profile:** Where along the backbone does inhibition most strongly attenuate or amplify the response?
- **Strength sweep:** Is the response monotone in \(g\), or are there non-monotone regimes?
- **Stability:** Does inhibition move the spectral abscissa toward or away from zero?
- **Transient gain:** Can inhibition reduce asymptotic instability while preserving or increasing transient amplification?
- **Node ablations:** Which inhibitory sources function as dynamical gates?
- **Edge ablations:** Which specific inhibitory projections have the greatest leverage?
- **Backbone geometry:** Are influential inhibitory edges preferentially feedforward, feedback, or long-range?
- **Null model:** Is the observed placement of inhibition special beyond inhibitory out-strength/degree?

### Important modeling caveats

1. A linear model is a first-order approximation; it does not include saturation, thresholds, refractory behavior, synaptic delays, or separate E/I population dynamics.
2. A negative matrix entry is an inhibitory **effective interaction** unless cell identity independently supports an inhibitory-neuron interpretation.
3. If neuron types are known, enforce Dale-consistent sign structure before making neuron-level claims.
4. For nonlinear follow-up, a Wilson–Cowan or firing-rate model is a natural next step.
5. If conduction/synaptic delays are important, replace the ODE with a delay differential equation or delayed neural-mass model.


## 12. Export key result tables

In [ ]:
RESULT_DIR = Path("connectome_inhibition_results")
RESULT_DIR.mkdir(exist_ok=True)

m1.to_csv(RESULT_DIR / "backbone_metrics_g1.csv", index=False)
sweep.to_csv(RESULT_DIR / "inhibition_sweep.csv", index=False)
transient.to_csv(RESULT_DIR / "transient_gain.csv", index=False)
node_ablation.to_csv(RESULT_DIR / "inhibitory_node_ablation.csv", index=False)
edge_ablation.to_csv(RESULT_DIR / "inhibitory_edge_ablation.csv", index=False)
motifs.to_csv(RESULT_DIR / "inhibitory_backbone_motifs.csv", index=False)

print("Saved result tables to:", RESULT_DIR)
